In [4]:
import numpy as np
import igl  # Ensure you have the libigl Python binding installed
import matplotlib.pyplot as plt
import os

def read_off(filename):
    """Reads a triangulated OFF file and returns vertices and faces."""
    with open(filename, 'r') as file:
        lines = file.readlines()
    # Remove empty lines and comments
    lines = [line.strip() for line in lines if line.strip() and not line.strip().startswith('#')]
    if lines[0] != 'OFF':
        raise ValueError("Not a valid OFF file: missing 'OFF' header.")
    # Second line: number of vertices, faces, and edges (edges not used)
    parts = lines[1].split()
    n_vertices, n_faces = int(parts[0]), int(parts[1])
    vertices = []
    for i in range(2, 2 + n_vertices):
        vertex = list(map(float, lines[i].split()))
        vertices.append(vertex)
    vertices = np.array(vertices)
    
    faces = []
    for i in range(2 + n_vertices, 2 + n_vertices + n_faces):
        parts = list(map(int, lines[i].split()))
        n_verts_face = parts[0]
        if n_verts_face != 3:
            raise ValueError("Found a non-triangular face. Please triangulate the mesh.")
        face_indices = parts[1:1 + n_verts_face]
        faces.append(face_indices)
    faces = np.array(faces)
    return vertices, faces

def compute_mesh_area_libigl(off_file):
    """Computes the total surface area of a triangulated mesh using libigl."""
    vertices, faces = read_off(off_file)
    # Compute per-face double areas (each value is 2 * area of the face)
    double_areas = igl.doublearea(vertices, faces)
    total_area = np.sum(double_areas) / 2.0
    return total_area

def read_log_file(filename):
    """
    Reads the simulation log file.
    Expects a header line starting with 'Iteration' followed by rows of numerical data.
    """
    header = None
    data = []
    with open(filename, 'r') as f:
        found_header = False
        for line in f:
            line = line.strip()
            if line.startswith("Iteration"):
                header = line.split()
                found_header = True
                continue
            if not found_header:
                continue
            try:
                row = list(map(float, line.split()))
                data.append(row)
            except Exception:
                continue
    data = np.array(data)
    return header, data

def main():
    # Windows file paths (using raw strings to avoid escape issues)
    logfile = r"C:\Users\didarula\Desktop\production_run\hauser_cube_prolate\z_axis\u=2.0\logfile.txt"
    off_file = r"C:\Users\didarula\Desktop\production_run\hauser_cube_prolate\z_axis\u=2.0\shifted_cube_prolate_z_axis.off"

    # Verify file existence
    if not os.path.exists(off_file):
        print("OFF file not found:", off_file)
        return
    if not os.path.exists(logfile):
        print("Log file not found:", logfile)
        return

    # Simulation parameter from log (particle adhesion strength)
    particle_adhesion_strength = 0.168827

    # Compute the particle area using libigl
    try:
        particle_area = compute_mesh_area_libigl(off_file)
    except Exception as e:
        print("Error reading OFF file with libigl:", e)
        return
    print(f"Particle area computed from '{off_file}': {particle_area}")

    # Read simulation log data
    header, data = read_log_file(logfile)
    if header is None:
        print("Header not found in the log file. Please check that the file contains a proper header.")
        return

    # Identify column indices for 'Time' and 'AdhesionEnergy'
    try:
        time_idx = header.index("Time")
        adhesion_energy_idx = header.index("AdhesionEnergy")
    except ValueError:
        print("Could not find 'Time' and/or 'AdhesionEnergy' in the log file header.")
        return

    # Extract time and adhesion energy from the data
    times = data[:, time_idx]
    adhesion_energies = data[:, adhesion_energy_idx]

    # Compute the wrapping fraction using:
    # wrapping_fraction = -(AdhesionEnergy) / (Adhesion Strength * particle_area)
    wrapping_fraction = -adhesion_energies / (particle_adhesion_strength * particle_area)

    # Plot time vs. wrapping fraction
    plt.figure(figsize=(8, 6))
    plt.plot(times, wrapping_fraction, marker='o', linestyle='-', color='blue')
    plt.xlabel("Time")
    plt.ylabel("Wrapping Fraction")
    plt.title("Time vs. Wrapping Fraction")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

if __name__ == '__main__':
    main()


ModuleNotFoundError: No module named 'igl'